# Detalle teórico y algebraico del ejemplo de prueba de hipótesis

Este notebook amplía el archivo `EjemploDePruebaDeHipotesisAplicadoAunSistemaDeComunicacionOptico.ipynb` con un enfoque más útil para preparar un parcial escrito.

Se apoya en dos fuentes de la carpeta `Teorico`:

- **Bixio Rimoldi, Principles of Digital Communication**.
- **Apuntes de asignatura - Comunicaciones Digitales - Augusto Cabrera**.

El objetivo no es solo mostrar el ejemplo, sino también dejar claro:

1. qué símbolos se usan;
2. qué ideas previas hay que manejar;
3. cómo se despejan las ecuaciones paso a paso;
4. cómo justificar en papel la regla de decisión y la probabilidad de error.

## 0. Qué conviene saber antes de empezar

Del apunte de Cabrera se desprende que, antes de este ejemplo, conviene tener claras estas ideas:

- **Fuente**: produce la hipótesis o mensaje $H$.
- **Transmisor**: mapea la hipótesis a una señal o situación física.
- **Canal**: introduce aleatoriedad.
- **Receptor**: observa una salida aleatoria $Y$ y decide $\hat H$.
- **Objetivo**: minimizar $P_e$ o, equivalentemente, maximizar $P_c$.

Para este notebook también conviene recordar:

- probabilidad condicional y regla de Bayes;
- diferencia entre variable aleatoria **discreta** y **continua**;
- propiedades de logaritmos;
- distribución de Poisson;
- interpretación de una desigualdad cuando la variable observada toma valores enteros.

Nota importante: en este ejemplo $Y$ es **discreta**, por eso la notación más prolija es $P_{Y\mid H}(y\mid i)$ y no $f_{Y\mid H}(y\mid i)$. El libro muchas veces usa una notación unificada para simplificar, pero para practicar en hoja conviene distinguirlo bien.

## 1. Simbología y notación

| Símbolo | Significado |
|---|---|
| $H$ | Hipótesis o mensaje transmitido. |
| $\hat H$ | Hipótesis decidida por el receptor. |
| $Y$ | Observable a la salida del canal. |
| $P_H(i)$ | Probabilidad a priori de la hipótesis $i$. |
| $P_{Y\mid H}(y\mid i)$ | Probabilidad condicional de observar $y$ si se transmitió $H=i$. |
| $P_{H\mid Y}(i\mid y)$ | Probabilidad a posteriori de la hipótesis $i$ al observar $y$. |
| $\Lambda(y)$ | Razón o cociente de verosimilitudes. |
| $\eta$ | Umbral de la prueba MAP binaria. |
| $\lambda_0, \lambda_1$ | Intensidades Poisson para $H=0$ y $H=1$. |
| $P_e$ | Probabilidad de error total. |
| $P_c$ | Probabilidad de decisión correcta. |
| $P_{FA}$ | Probabilidad de falsa alarma. |
| $P_{MD}$ | Probabilidad de miss detection o error de omisión. |

Relaciones básicas:

$$P_e = P(\hat H \neq H), \qquad P_c = P(\hat H = H), \qquad P_e = 1 - P_c$$

En el apunte de Cabrera también aparece una idea importante para estudiar: **detección**, **decodificación**, **prueba de hipótesis** y **toma de decisiones** se usan como conceptos muy cercanos dentro del problema del receptor.

## 2. Mapa teórico de las fuentes

| Tema | Libro / Apunte | Por qué importa acá |
|---|---|---|
| Modelo fuente-transmisor-canal-receptor | Apunte Cabrera, Unidad 2, Introducción | Da el marco general del problema. |
| Prueba de hipótesis | Rimoldi, Cap. 2 Sec. 2.2 y Apunte Cabrera, Sec. 1.3 | Define qué problema resuelve el receptor. |
| Ejemplo óptico Poisson | Rimoldi, Ejemplo 2.4 y Apunte Cabrera, Sec. 1.3 | Justifica el modelo con fotones y LED. |
| Hipótesis binaria | Rimoldi, Sec. 2.2.1 y Apunte Cabrera, Sec. 1.4 | Lleva a la comparación entre dos alternativas. |
| MAP y ML | Ambas fuentes | Explican por qué se compara posterior o verosimilitud. |
| Estadístico suficiente | Rimoldi, Sec. 2.5 y Apunte Cabrera, Unidad 3 | Da sentido a usar una suma o proyección en lugar de guardar toda la observación. |
| Combinaciones lineales | Rimoldi, Sec. 2.4.2 y Unidad 3 del apunte | Conecta este ejemplo con receptores más generales. |

El foco del nuevo notebook sigue siendo el ejemplo Poisson, pero ahora con la estructura conceptual que aparece más explícita en el apunte.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import factorial, exp

plt.style.use("seaborn-v0_8-whitegrid")
rng = np.random.default_rng(12345)

def poisson_pmf(y, lam):
    y = np.asarray(y, dtype=int)
    return np.array([(lam ** int(k)) * exp(-lam) / factorial(int(k)) for k in y], dtype=float)

def threshold_map_poisson(lambda0, lambda1, p0=0.5, p1=0.5):
    return ((lambda1 - lambda0) + np.log(p0 / p1)) / np.log(lambda1 / lambda0)

def exact_error_poisson(lambda0, lambda1, gamma, p0=0.5, p1=0.5, y_max=40):
    ys = np.arange(0, y_max + 1)
    py_h0 = poisson_pmf(ys, lambda0)
    py_h1 = poisson_pmf(ys, lambda1)
    pfa = py_h0[ys >= gamma].sum()
    pmd = py_h1[ys < gamma].sum()
    pe = p0 * pfa + p1 * pmd
    return pfa, pmd, pe

def simulate_binary_poisson(lambda0, lambda1, samples=100000, seed=0):
    local_rng = np.random.default_rng(seed)
    h = local_rng.integers(0, 2, size=samples)
    y = np.where(h == 0, local_rng.poisson(lambda0, size=samples), local_rng.poisson(lambda1, size=samples))
    return h, y

def simulate_blocks(lambda0, lambda1, n_obs=5, blocks=80000, seed=123):
    local_rng = np.random.default_rng(seed)
    h = local_rng.integers(0, 2, size=blocks)
    y_blocks = np.where(
        h[:, None] == 0,
        local_rng.poisson(lambda0, size=(blocks, n_obs)),
        local_rng.poisson(lambda1, size=(blocks, n_obs))
    )
    return h, y_blocks


## 3. Modelo del problema: qué hace cada bloque

Siguiendo el enfoque del apunte de Cabrera:

- **Fuente**: genera $H \in \{0,1\}$.
- **Transmisor**: si $H=1$, enciende el LED; si $H=0$, lo deja apagado.
- **Canal + fotodetector**: producen una observación aleatoria $Y$, que es el número de fotones detectados.
- **Receptor**: a partir del valor observado $y$, decide si la hipótesis transmitida fue $0$ o $1$.

En este ejemplo, la incertidumbre no viene de un ruido gaussiano sino del hecho de que estamos contando fotones, y esos conteos se modelan con Poisson.

## 4. Distribución de Poisson y significado físico

El **Ejemplo 2.4** del libro y el apunte de Cabrera usan este modelo:

$$Y|H=0 \sim \text{Poisson}(\lambda_0), \qquad Y|H=1 \sim \text{Poisson}(\lambda_1), \qquad 0 \leq \lambda_0 < \lambda_1$$

La pmf de una Poisson es:

$$P(Y=y) = \frac{\lambda^y}{y!} e^{-\lambda}, \qquad y = 0,1,2,\dots$$

Además,

$$E[Y] = \lambda, \qquad Var(Y) = \lambda$$

Interpretación física:

- $\lambda_0$ representa la intensidad media cuando el LED está apagado; puede haber fotones igual por luz ambiente.
- $\lambda_1$ representa la intensidad media cuando el LED está encendido.
- cuanto más separados estén $\lambda_0$ y $\lambda_1$, más fácil será decidir.

In [ ]:
lambda_demo = 2
samples_demo = rng.poisson(lambda_demo, size=100000)
support_demo = np.arange(0, 12)
pmf_demo = poisson_pmf(support_demo, lambda_demo)

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].bar(support_demo, pmf_demo, color="#4C78A8")
ax[0].set_title(f"PMF teórica de Poisson con lambda = {lambda_demo}")
ax[0].set_xlabel("y")
ax[0].set_ylabel("P(Y=y)")

ax[1].hist(samples_demo, bins=np.arange(-0.5, 12.5, 1), density=True, color="#F58518", alpha=0.8)
ax[1].set_title("Histograma Monte Carlo")
ax[1].set_xlabel("y")
ax[1].set_ylabel("Frecuencia relativa")

plt.tight_layout()
plt.show()

print(f"Media muestral: {samples_demo.mean():.4f}")
print(f"Varianza muestral: {samples_demo.var():.4f}")

## 5. De Bayes a la regla MAP, paso a paso

Queremos decidir qué hipótesis es más creíble después de observar $Y=y$.

Por Bayes:

$$P_{H|Y}(i|y) = \frac{P_H(i) P_{Y|H}(y|i)}{P_Y(y)}$$

La regla óptima general elige la hipótesis con mayor probabilidad a posteriori:

$$\hat H(y) = \arg\max_i P_{H|Y}(i|y)$$

Reemplazando Bayes:

$$\hat H(y) = \arg\max_i \frac{P_H(i) P_{Y|H}(y|i)}{P_Y(y)}$$

Como $P_Y(y)$ no depende de $i$, no cambia el máximo. Entonces:

$$\hat H(y) = \arg\max_i P_H(i) P_{Y|H}(y|i)$$

Esta es la regla **MAP**.

Si las hipótesis son equiprobables, por ejemplo $P_H(0)=P_H(1)=1/2$, entonces el factor $P_H(i)$ es el mismo para todas las hipótesis y también puede ignorarse:

$$\hat H(y) = \arg\max_i P_{Y|H}(y|i)$$

Esta es la regla **ML**.

Resumen conceptual:

- **MAP** usa observación + prior.
- **ML** usa solo la verosimilitud.
- Si los prior son uniformes, $MAP = ML$.

## 6. Prueba de hipótesis binaria: despeje completo

En el caso binario $H \in \{0,1\}$, la regla MAP puede escribirse así:

$$\hat H = 1 \quad \text{si} \quad P_{H|Y}(1|y) \ge P_{H|Y}(0|y)$$

Usando Bayes:

$$\hat H = 1 \quad \text{si} \quad \frac{P_H(1) P_{Y|H}(y|1)}{P_Y(y)} \ge \frac{P_H(0) P_{Y|H}(y|0)}{P_Y(y)}$$

Como $P_Y(y) > 0$, se cancela en ambos lados:

$$\hat H = 1 \quad \text{si} \quad P_H(1) P_{Y|H}(y|1) \ge P_H(0) P_{Y|H}(y|0)$$

Ahora dividimos ambos lados por la cantidad positiva $P_H(1) P_{Y\mid H}(y\mid 0)$:

$$\hat H = 1 \quad \text{si} \quad \frac{P_{Y|H}(y|1)}{P_{Y|H}(y|0)} \ge \frac{P_H(0)}{P_H(1)}$$

Definimos:

$$\Lambda(y) = \frac{P_{Y|H}(y|1)}{P_{Y|H}(y|0)}, \qquad \eta = \frac{P_H(0)}{P_H(1)}$$

Entonces la prueba MAP binaria queda:

$$\Lambda(y) \overset{\hat H = 1}{\underset{\hat H = 0}{\gtreqless}} \eta$$

Y si $P_H(0)=P_H(1)$, entonces $\eta = 1$ y la prueba MAP se convierte en ML:

$$\Lambda(y) \overset{\hat H = 1}{\underset{\hat H = 0}{\gtreqless}} 1$$

Esta forma es importante para rendir porque es la puerta de entrada al despeje del umbral.

## 7. Aplicación al caso Poisson: derivación completa del umbral

Tomamos el caso equiprobable para usar ML:

$$P_{Y|H}(y|1) \overset{\hat H = 1}{\underset{\hat H = 0}{\gtreqless}} P_{Y|H}(y|0)$$

Sustituyendo las pmf Poisson:

$$\frac{\lambda_1^y}{y!} e^{-\lambda_1} \overset{\hat H = 1}{\underset{\hat H = 0}{\gtreqless}} \frac{\lambda_0^y}{y!} e^{-\lambda_0}$$

Paso 1. Cancelamos $y!$ porque aparece en ambos miembros:

$$\lambda_1^y e^{-\lambda_1} \overset{\hat H = 1}{\underset{\hat H = 0}{\gtreqless}} \lambda_0^y e^{-\lambda_0}$$

Paso 2. Dividimos ambos lados por la cantidad positiva $\lambda_0^y e^{-\lambda_1}$:

$$\left(\frac{\lambda_1}{\lambda_0}\right)^y \overset{\hat H = 1}{\underset{\hat H = 0}{\gtreqless}} e^{\lambda_1 - \lambda_0}$$

Paso 3. Aplicamos logaritmo natural a ambos lados. Esto es válido porque $\ln(\cdot)$ es creciente:

$$\ln\left[\left(\frac{\lambda_1}{\lambda_0}\right)^y\right] \overset{\hat H = 1}{\underset{\hat H = 0}{\gtreqless}} \ln\left(e^{\lambda_1 - \lambda_0}\right)$$

Paso 4. Usamos $\ln(a^b)=b\ln(a)$ y $\ln(e^x)=x$:

$$y \ln\left(\frac{\lambda_1}{\lambda_0}\right) \overset{\hat H = 1}{\underset{\hat H = 0}{\gtreqless}} \lambda_1 - \lambda_0$$

Paso 5. Como $\lambda_1 > \lambda_0$, entonces $\ln(\lambda_1/\lambda_0) > 0$, y podemos dividir sin cambiar el sentido de la desigualdad:

$$y \overset{\hat H = 1}{\underset{\hat H = 0}{\gtreqless}} \frac{\lambda_1 - \lambda_0}{\ln(\lambda_1/\lambda_0)}$$

Ese valor es el **umbral teórico real**.

Si además usamos MAP con prior no uniformes, el despeje completo queda:

$$\frac{P_{Y|H}(y|1)}{P_{Y|H}(y|0)} \overset{\hat H = 1}{\underset{\hat H = 0}{\gtreqless}} \frac{P_H(0)}{P_H(1)}$$

$$\frac{\lambda_1^y e^{-\lambda_1}}{\lambda_0^y e^{-\lambda_0}} \overset{\hat H = 1}{\underset{\hat H = 0}{\gtreqless}} \frac{P_H(0)}{P_H(1)}$$

$$y \ln\left(\frac{\lambda_1}{\lambda_0}\right) - (\lambda_1 - \lambda_0) \overset{\hat H = 1}{\underset{\hat H = 0}{\gtreqless}} \ln\left(\frac{P_H(0)}{P_H(1)}\right)$$

$$y \overset{\hat H = 1}{\underset{\hat H = 0}{\gtreqless}} \frac{(\lambda_1 - \lambda_0) + \ln(P_H(0)/P_H(1))}{\ln(\lambda_1/\lambda_0)}$$

Detalle clave para parcial: como $Y$ toma valores enteros, si el umbral teórico da $1.82$, la regla práctica es:

$$Y \ge 1.82 \iff Y \ge 2$$

O sea, el umbral implementable es $\gamma = \lceil 1.82 \rceil = 2$.

In [ ]:
lambda0 = 1
lambda1 = 3
p0 = 0.5
p1 = 0.5

threshold_ml = threshold_map_poisson(lambda0, lambda1, p0, p1)
threshold_map_biased = threshold_map_poisson(lambda0, lambda1, p0=0.7, p1=0.3)
gamma_ml = int(np.ceil(threshold_ml))

print(f"Umbral teórico real para ML: {threshold_ml:.4f}")
print(f"Umbral entero implementable: gamma = {gamma_ml}")
print(f"Umbral teórico para MAP con P(H=0)=0.7 y P(H=1)=0.3: {threshold_map_biased:.4f}")

In [ ]:
support = np.arange(0, 11)
pmf_h0 = poisson_pmf(support, lambda0)
pmf_h1 = poisson_pmf(support, lambda1)

plt.figure(figsize=(12, 4))
plt.bar(support - 0.15, pmf_h0, width=0.3, label=r"$P_{Y|H}(y|0)$", color="#54A24B")
plt.bar(support + 0.15, pmf_h1, width=0.3, label=r"$P_{Y|H}(y|1)$", color="#E45756")
plt.axvline(threshold_ml, color="black", linestyle="--", label=f"umbral teorico = {threshold_ml:.2f}")
plt.axvline(gamma_ml - 0.5, color="#4C78A8", linestyle=":", label=f"frontera discreta en y >= {gamma_ml}")
plt.xticks(support)
plt.xlabel("y")
plt.ylabel("Probabilidad")
plt.title("Distribuciones condicionadas y umbral de decision")
plt.legend()
plt.show()

## 8. Probabilidad de error exacta: fórmula para resolver en hoja

Como $Y$ es discreta, la probabilidad de error se calcula con **sumatorias**, no con integrales.

Si la regla de decisión es:

$$\hat H = 1 \quad \text{si} \quad Y \ge \gamma, \qquad \hat H = 0 \quad \text{si} \quad Y < \gamma$$

entonces:

### 8.1. Falsa alarma

Es decidir $1$ cuando en realidad se transmitió $0$:

$$P_{FA} = P(\hat H = 1 | H=0) = P(Y \ge \gamma | H=0)$$

Como $Y$ es discreta:

$$P_{FA} = \sum_{y=\gamma}^{\infty} P_{Y|H}(y|0) = \sum_{y=\gamma}^{\infty} \frac{\lambda_0^y}{y!} e^{-\lambda_0}$$

### 8.2. Miss detection u omisión

Es decidir $0$ cuando en realidad se transmitió $1$:

$$P_{MD} = P(\hat H = 0 | H=1) = P(Y < \gamma | H=1)$$

Como $Y$ es discreta:

$$P_{MD} = \sum_{y=0}^{\gamma-1} P_{Y|H}(y|1) = \sum_{y=0}^{\gamma-1} \frac{\lambda_1^y}{y!} e^{-\lambda_1}$$

### 8.3. Probabilidad de error total

$$P_e = P_H(0) P_{FA} + P_H(1) P_{MD}$$

Si las hipótesis son equiprobables:

$$P_e = \frac{1}{2} P_{FA} + \frac{1}{2} P_{MD}$$

Esta es una de las fórmulas más importantes para practicar en papel.

In [ ]:
pfa_exact, pmd_exact, pe_exact = exact_error_poisson(lambda0, lambda1, gamma_ml, p0=0.5, p1=0.5)

print(f"PFA exacta para gamma={gamma_ml}: {pfa_exact:.6f}")
print(f"PMD exacta para gamma={gamma_ml}: {pmd_exact:.6f}")
print(f"Pe exacta para gamma={gamma_ml}: {pe_exact:.6f}")

In [ ]:
h, y = simulate_binary_poisson(lambda0, lambda1, samples=200000, seed=7)
candidate_gammas = np.arange(0, 9)

pe_exact_values = []
pe_sim_values = []
pfa_values = []
pmd_values = []

for gamma in candidate_gammas:
    h_hat = (y >= gamma).astype(int)
    pe_sim_values.append(np.mean(h_hat != h))
    pfa_tmp, pmd_tmp, pe_tmp = exact_error_poisson(lambda0, lambda1, gamma, p0=0.5, p1=0.5)
    pe_exact_values.append(pe_tmp)
    pfa_values.append(pfa_tmp)
    pmd_values.append(pmd_tmp)

best_gamma = int(candidate_gammas[np.argmin(pe_exact_values)])

plt.figure(figsize=(12, 4))
plt.plot(candidate_gammas, pe_exact_values, marker="o", label="Pe exacta")
plt.plot(candidate_gammas, pe_sim_values, marker="s", label="Pe simulada")
plt.plot(candidate_gammas, pfa_values, marker="^", label="PFA exacta")
plt.plot(candidate_gammas, pmd_values, marker="v", label="PMD exacta")
plt.axvline(best_gamma, color="#B279A2", linestyle=":", label=f"mejor gamma = {best_gamma}")
plt.xlabel("Umbral entero gamma")
plt.ylabel("Probabilidad")
plt.title("Desempeño del receptor en función del umbral discreto")
plt.legend()
plt.show()

print(f"Mejor umbral entero según Pe exacta: gamma = {best_gamma}")
print(f"Pe exacta mínima: {min(pe_exact_values):.6f}")

## 9. Combinaciones lineales y estadístico suficiente

Este punto conviene estudiarlo porque conecta directamente con lo que después se usa en receptores AWGN y en laboratorio.

Supongamos que para una misma hipótesis no observamos un único conteo, sino $n$ conteos independientes:

$$(Y_1, Y_2, \dots, Y_n)$$

Si condicionamos por la hipótesis y suponemos independencia, entonces:

$$P_{\mathbf Y|H}(\mathbf y|i) = \prod_{k=1}^{n} P_{Y_k|H}(y_k|i)$$

La razón de verosimilitudes queda:

$$\Lambda(\mathbf y) = \frac{P_{\mathbf Y|H}(\mathbf y|1)}{P_{\mathbf Y|H}(\mathbf y|0)} = \prod_{k=1}^{n} \frac{P_{Y_k|H}(y_k|1)}{P_{Y_k|H}(y_k|0)}$$

Tomamos logaritmo:

$$\ln \Lambda(\mathbf y) = \sum_{k=1}^{n} \ln\left(\frac{P_{Y_k|H}(y_k|1)}{P_{Y_k|H}(y_k|0)}\right)$$

Ahora reemplazamos las pmf Poisson:

$$\ln \Lambda(\mathbf y) = \sum_{k=1}^{n} \ln\left(\frac{\lambda_1^{y_k} e^{-\lambda_1}/y_k!}{\lambda_0^{y_k} e^{-\lambda_0}/y_k!}\right)$$

Se cancela $y_k!$ dentro de cada cociente:

$$\ln \Lambda(\mathbf y) = \sum_{k=1}^{n} \ln\left(\left(\frac{\lambda_1}{\lambda_0}\right)^{y_k} e^{-(\lambda_1-\lambda_0)}\right)$$

Usando logaritmos:

$$\ln \Lambda(\mathbf y) = \sum_{k=1}^{n} \left[y_k \ln\left(\frac{\lambda_1}{\lambda_0}\right) - (\lambda_1 - \lambda_0)\right]$$

Distribuyendo la sumatoria:

$$\ln \Lambda(\mathbf y) = \left(\sum_{k=1}^{n} y_k\right) \ln\left(\frac{\lambda_1}{\lambda_0}\right) - n(\lambda_1 - \lambda_0)$$

Definimos:

$$S = \sum_{k=1}^{n} y_k$$

Entonces la decisión depende del vector completo $\mathbf y$ solamente a través de $S$.

Conclusión:

- $S$ es una **combinación lineal** de observaciones;
- $S$ es un **estadístico suficiente** para decidir;
- esta misma lógica reaparece en el caso AWGN como correlación, producto interno o proyección.

In [ ]:
n_obs = 5
h_blocks, y_blocks = simulate_blocks(lambda0, lambda1, n_obs=n_obs, blocks=100000, seed=21)

sum_stat = y_blocks.sum(axis=1)
llr = sum_stat * np.log(lambda1 / lambda0) - n_obs * (lambda1 - lambda0)

h_hat_llr = (llr >= 0).astype(int)
sum_threshold = n_obs * threshold_ml
h_hat_sum = (sum_stat >= sum_threshold).astype(int)

agreement = np.mean(h_hat_llr == h_hat_sum)
pe_blocks = np.mean(h_hat_sum != h_blocks)

print(f"Cantidad de observaciones por decision: {n_obs}")
print(f"Umbral teorico sobre la suma: {sum_threshold:.4f}")
print(f"Coincidencia entre decision por LLR y por suma: {agreement:.4f}")
print(f"Probabilidad de error usando la suma como estadistico suficiente: {pe_blocks:.4f}")

n_grid = np.arange(1, 11)
pe_n = []

for n_tmp in n_grid:
    h_n, y_n = simulate_blocks(lambda0, lambda1, n_obs=n_tmp, blocks=40000, seed=100 + n_tmp)
    stat_n = y_n.sum(axis=1)
    thr_n = n_tmp * threshold_ml
    h_hat_n = (stat_n >= thr_n).astype(int)
    pe_n.append(np.mean(h_hat_n != h_n))

plt.figure(figsize=(10, 4))
plt.plot(n_grid, pe_n, marker="o", color="#4C78A8")
plt.xlabel("Cantidad de observaciones independientes por decision")
plt.ylabel("Probabilidad de error")
plt.title("Mas observaciones implican menor probabilidad de error")
plt.show()

## 10. Qué parte conecta con AWGN, correladores y SDR

El apunte de Cabrera pone mucho foco en el receptor AWGN discreto y continuo, en estadísticos suficientes y en correladores. Vale la pena dejar la conexión explícita:

- en este ejemplo Poisson, el receptor decide comparando un **conteo** o una **suma de conteos**;
- en AWGN discreto, el receptor termina comparando una **proyección**, un **producto interno** o una **distancia**;
- en AWGN continuo, el correlador y el filtro adaptado implementan esa reducción a una estadística suficiente;
- en laboratorio y en SDR, esta filosofía se mantiene: medir, filtrar, proyectar, resumir y decidir.

Por eso este ejemplo no está aislado: es la forma más simple de la misma lógica de diseño de receptor.

In [ ]:
lambda1_grid = np.array([2, 3, 5, 8, 12, 20])
intensidad_db = 10 * np.log10(lambda0 + lambda1_grid)

def block_error_vs_lambda1(lambda0, lambda1_values, n_obs=1, blocks=40000):
    pe = []
    for idx, lam1 in enumerate(lambda1_values):
        h_tmp, y_tmp = simulate_blocks(lambda0, lam1, n_obs=n_obs, blocks=blocks, seed=500 + idx + n_obs)
        stat_tmp = y_tmp.sum(axis=1)
        thr_tmp = n_obs * threshold_map_poisson(lambda0, lam1)
        h_hat_tmp = (stat_tmp >= thr_tmp).astype(int)
        pe.append(np.mean(h_hat_tmp != h_tmp))
    return np.array(pe)

pe_n1 = block_error_vs_lambda1(lambda0, lambda1_grid, n_obs=1)
pe_n5 = block_error_vs_lambda1(lambda0, lambda1_grid, n_obs=5)

plt.figure(figsize=(10, 4))
plt.semilogy(intensidad_db, pe_n1, marker="o", label="1 observacion")
plt.semilogy(intensidad_db, pe_n5, marker="s", label="5 observaciones")
plt.xlabel("Intensidad promedio (dB)")
plt.ylabel("Probabilidad de error")
plt.title("Mejor separacion entre hipotesis implica menor error")
plt.legend()
plt.show()

## 11. Guía de estudio para practicar en hoja

Si esto te lo tomaran en un parcial escrito, el recorrido recomendable sería:

1. Definir claramente $H$, $Y$, $\hat H$, $P_e$ y el modelo condicional.
2. Aclarar si el problema es MAP o ML.
3. Escribir la desigualdad binaria entre hipótesis.
4. Reemplazar las pmf o pdf correspondientes.
5. Cancelar términos comunes.
6. Aplicar logaritmos si hace falta.
7. Despejar la variable de decisión.
8. Ajustar el resultado si la variable observada es discreta.
9. Escribir $P_{FA}$, $P_{MD}$ y $P_e$ con sumatorias o integrales según el caso.

Identidades algebraicas útiles:

$$\ln\left(\frac{a}{b}\right)=\ln a - \ln b, \qquad \ln(a^n)=n\ln a, \qquad \ln(e^x)=x$$

Observación práctica:

- En este ejemplo **no** hace falta la función $Q$ porque no estamos en un caso gaussiano escalar.
- Sí conviene estudiarla después, porque en el apunte aparece enseguida cuando se pasa al canal AWGN.

## 12. Mini ejercicios para practicar

1. Repetir el despeje del umbral para $\lambda_0=2$ y $\lambda_1=5$ con hipótesis equiprobables.
2. Repetir el despeje anterior, pero con $P(H=0)=0.8$ y $P(H=1)=0.2$.
3. Justificar por qué un umbral teórico $\theta=2.14$ se implementa como $Y \ge 3$.
4. Escribir $P_{FA}$, $P_{MD}$ y $P_e$ exactas para $\gamma=3$.
5. Partiendo de $n$ observaciones independientes, volver a demostrar que la decisión depende de $S=\sum_k Y_k$.

Si podés hacer esos cinco ejercicios prolijos en hoja, ya te queda bastante firme la mecánica algebraica de este tema.